In [36]:
import pandas as pd
import folium

In [2]:
# Import Data
df = pd.read_csv("./cleaned.csv")
df = df.drop("Unnamed: 0", axis=1)


In [33]:
df_group = df.groupby("location").count()
df_cities = pd.DataFrame()
df_group.reset_index(inplace=True)
df_cities["city"] = df_group["location"]
df_cities["num_jobs"] = df_group["job_title"]
i = 0
df_cities["top_company"] = ""
df_cities["top_job_title"] = ""
for city in df_cities["city"]:
    df_cities["top_company"].iloc[i] = df[df["location"]==city]["company_name"].value_counts().reset_index()["index"].iloc[0]
    df_cities["top_job_title"].iloc[i] = df[df["location"]==city]["clean_job_title"].value_counts().reset_index()["index"].iloc[0]
    i += 1
df_cities    

/var/folders/m0/y_433zr10m79zw6c3ccqn31c0000gn/T/ipykernel_66170/3195251715.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cities["top_company"].iloc[i] = df[df["location"]==city]["company_name"].value_counts().reset_index()["index"].iloc[0]
/var/folders/m0/y_433zr10m79zw6c3ccqn31c0000gn/T/ipykernel_66170/3195251715.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cities["top_job_title"].iloc[i] = df[df["location"]==city]["clean_job_title"].value_counts().reset_index()["index"].iloc[0]
/var/folders/m0/y_433zr10m79zw6c3ccqn31c0000gn/T/ipykernel_66170/3195251715.py:10: SettingWithCopyWarning: 
A value is 

,city,num_jobs,top_company,top_job_title
0,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer
1,"Ann Arbor, MI",1,University of Michigan,health natural language processing research sp...
2,"Annapolis Junction, MD",12,"Erias Ventures, LLC",cloud software engineer
3,"Annapolis, MD",6,Raytheon Technologies,cloud software engineer
4,"Arden Hills, MN",1,LandOLakes,Data Scientist
...,...,...,...,...
152,"Westborough, MA",1,Experfy,Data Scientist
153,"Wheaton-Glenmont, MD",1,Guidehouse,Data Scientist
154,"White Plains, NY",1,Trigyn,scrum master/business analyst (cloud computing...
155,"Wilmington, DE",1,JPMorgan Chase,"lead software engineer, cloud and big data"


In [42]:
df_cities

,city,num_jobs,top_company,top_job_title
0,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer
1,"Ann Arbor, MI",1,University of Michigan,health natural language processing research sp...
2,"Annapolis Junction, MD",12,"Erias Ventures, LLC",cloud software engineer
3,"Annapolis, MD",6,Raytheon Technologies,cloud software engineer
4,"Arden Hills, MN",1,LandOLakes,Data Scientist
...,...,...,...,...
152,"Westborough, MA",1,Experfy,Data Scientist
153,"Wheaton-Glenmont, MD",1,Guidehouse,Data Scientist
154,"White Plains, NY",1,Trigyn,scrum master/business analyst (cloud computing...
155,"Wilmington, DE",1,JPMorgan Chase,"lead software engineer, cloud and big data"


In [45]:
df_cities = df_cities.merge(df[["location_coord", "location"]], left_on = "city", right_on = "location")

In [47]:
# filter data to remove USA as location
df_cities = df_cities[df["location"] != "United States"]
df_cities = df_cities.dropna(subset=["location_coord"])

# change coordinates from string to floats
df_cities["location_coord"] = [eval(x) for x in df_cities["location_coord"]]

/var/folders/m0/y_433zr10m79zw6c3ccqn31c0000gn/T/ipykernel_66170/3354436520.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df_cities = df_cities[df["location"] != "United States"]


In [48]:
df_cities

,city,num_jobs,top_company,top_job_title,location_coord,location
2,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer,"(38.8051095, -77.0470229)","Alexandria, VA"
3,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer,"(38.8051095, -77.0470229)","Alexandria, VA"
4,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer,"(38.8051095, -77.0470229)","Alexandria, VA"
5,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer,"(38.8051095, -77.0470229)","Alexandria, VA"
6,"Alexandria, VA",8,Booz Allen Hamilton,Machine Learning Engineer,"(38.8051095, -77.0470229)","Alexandria, VA"
...,...,...,...,...,...,...
587,"Weehawken, NJ",1,HexaQuest Global,java springboot cloud developer,"(40.7695457, -74.0204177)","Weehawken, NJ"
588,"West Columbia, SC",1,Harsco Corporation,Machine Learning Engineer,"(33.9934867, -81.0739826)","West Columbia, SC"
590,"Wheaton-Glenmont, MD",1,Guidehouse,Data Scientist,"(39.04979325, -77.05228508396917)","Wheaton-Glenmont, MD"
591,"White Plains, NY",1,Trigyn,scrum master/business analyst (cloud computing...,"(41.0339862, -73.7629097)","White Plains, NY"


In [34]:
def popup_table(i):
    # Define variables needed
    location = df_cities["city"].iloc[i]
    number_of_jobs = df_cities["num_jobs"].iloc[i]
    company = df_cities["top_company"].iloc[i]
    job_title = df_cities["top_job_title"].iloc[i]
    # Define html that combines a table to information next to the bar chart for each county
    html =""" <!DOCTYPE html>
<html>
<head>
<h4>{}</h4>""".format(location) + """
</head>
<body>
<table>
<tr>
<td>
<table style="width: 250px;color: black;">
<tbody>
<tr>
<th style="background-color: #CCCCCC" ;"><span style="color: black;">Number of Job Postings</span>
</td>
<td style="width: 75px;text-align: center;background-color: #CCCCCC" ;">{}</td>""".format(number_of_jobs) + """
</tr>
<tr>
<th style="background-color: #e3e3e3" ;"><span style="color: black;">Top Company</span></td>
<td style="width: 75px;text-align: center;background-color: #e3e3e3" ;">{}</td>""".format(company) + """
</tr>
<tr>
<th style="background-color: #CCCCCC" ;"><span style="color: black">Top Job Title</span></td>
<td style="width: 75px;text-align: center;background-color: #CCCCCC" ;">{}</td>""".format(job_title) + """
</tr>
</tbody>
</table>
</td>
</tr>
</table>
</body>
</html>
"""
    return html

In [65]:
loc_dict = {}
loc_dict["Washington, DC"] = [39,-77]
loc_dict["USA"] = [40,-95]
loc_dict["New York, NY"] = [40.75,-74]
loc_dict["Boston, MA"] = [42.5,-71.2]
loc_dict["San Francisco, CA"] = [37.75,-122]



[39, -77]

In [93]:
# Initialize Folium Map centered on USA
fig = folium.Map(location=[37.75,-122], tiles="OpenStreetMap", zoom_start=9)
# Define empty datagroup 
data_group = folium.FeatureGroup(name='Data')

# for each observation (bubble on plot)
for i in range(0, len(df_cities)):
    html = popup_table(i) # run html function defined above to get html output for that county
    data_group.add_child(folium.Marker( # add bubble plot
        location=df_cities["location_coord"].iloc[i], # set bubble at longitude and latitude
        popup= folium.Popup(folium.Html(html, script = True)), # add html output to folium popup
        fill=True,
        weight=3,
        opacity=1,
        fillopacity=0.9,
        ))
fig.add_child(data_group) # add datagroup to figure
fig